# 🌸 Lab 3: Iris Flower Classification

**Challenge**: Classify iris flowers into 3 species based on petal and sepal measurements.

**What you'll use**: pandas, scikit-learn, KNN, SVM, visualization

---
### The Story
A botanist wants to automatically identify iris flower species from measurements. There are 3 species: Setosa, Versicolor, and Virginica. You'll compare multiple classifiers and pick the best one!

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

print('✅ Ready!')

In [ ]:
# Load the Iris dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = [iris.target_names[t] for t in iris.target]
df['target'] = iris.target

print('Classes:', iris.target_names)
print('Samples per class:')
print(df['species'].value_counts())
df.head()

In [ ]:
# Visualize: Can we see the 3 species?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'setosa': '#ff6b6b', 'versicolor': '#4ecdc4', 'virginica': '#00ff9d'}
for species, color in colors.items():
    subset = df[df['species'] == species]
    axes[0].scatter(subset['sepal length (cm)'], subset['sepal width (cm)'], 
                    label=species, color=color, alpha=0.7)
    axes[1].scatter(subset['petal length (cm)'], subset['petal width (cm)'], 
                    label=species, color=color, alpha=0.7)

axes[0].set_title('Sepal: Length vs Width'); axes[0].legend()
axes[1].set_title('Petal: Length vs Width'); axes[1].legend()
plt.tight_layout()
plt.show()
print('Notice: Petal measurements separate the species much better!')

In [ ]:
# Prepare data
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# YOUR TURN: Train a KNN classifier
# Try k=5 first (5 nearest neighbors)
# Hint: KNeighborsClassifier(n_neighbors=5)

knn = ___  # ← Create KNN with n_neighbors=5
___  # ← Fit on X_train_s, y_train

knn_preds = knn.predict(X_test_s)
knn_acc = accuracy_score(y_test, knn_preds)
print(f'KNN Accuracy: {knn_acc:.1%}')

In [ ]:
# YOUR TURN: Find the best K value
# Try k from 1 to 15 and find which gives the best accuracy

k_values = range(1, 16)
accuracies = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_s))
    accuracies.append(acc)

best_k = ___  # ← Find the k with the highest accuracy
# Hint: k_values[accuracies.index(max(accuracies))]

plt.plot(k_values, accuracies, 'o-', color='#00ff9d')
plt.xlabel('K value'); plt.ylabel('Accuracy')
plt.title('KNN: Finding the Best K')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.legend()
plt.show()
print(f'Best K: {best_k}, Accuracy: {max(accuracies):.1%}')

In [ ]:
# Train final KNN with best_k and also an SVM
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_s, y_train)
final_knn_acc = accuracy_score(y_test, final_knn.predict(X_test_s))

# SVM with RBF kernel
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train_s, y_train)
svm_acc = accuracy_score(y_test, svm.predict(X_test_s))

print(f'Best KNN (k={best_k}) Accuracy: {final_knn_acc:.1%}')
print(f'SVM Accuracy: {svm_acc:.1%}')
print('\nDetailed Report (KNN):')
print(classification_report(y_test, final_knn.predict(X_test_s), target_names=iris.target_names))

In [ ]:
# ✅ TEST CELL
assert hasattr(knn, 'predict'), 'knn must be trained'
assert knn_acc >= 0.90, f'KNN accuracy should be >= 90%, got {knn_acc:.1%}'
assert isinstance(best_k, (int, np.integer)), 'best_k must be an integer'
assert 1 <= best_k <= 15, 'best_k must be between 1 and 15'
print('🎉 All tests passed!')
print(f'KNN: {knn_acc:.1%} | SVM: {svm_acc:.1%}')